In [13]:
from nlp4bia.datasets.Dataset import BenchmarkDataset
from nlp4bia.datasets import config
from nlp4bia.datasets.utils import handlers

import os
        
from requests import get
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

class Drugtemist(BenchmarkDataset):
    URL = "https://zenodo.org/records/11368861/files/multicardioner_train+dev+test+bg+mappings_240528.zip?download=1"
    NAME = "multicardioner_train+dev+test+bg+mappings_240528"
    DS_COLUMNS = config.DS_COLUMNS
    
    def __init__(self, lang="es", path=None, name=NAME, url=URL, download_if_missing=True):
        super().__init__(lang, name, path, url, download_if_missing)

    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        
        train_path = os.path.join(self.path, "track2/drugtemist_train/es/tsv/multicardioner_track2_drugtemist_train_es.tsv")
        texts_train_path = os.path.join(self.path, "track2/drugtemist_train/es/brat/")
        # test_path = os.path.join(self.path, "medprocner_test/tsv/medprocner_tsv_test_subtask2.tsv")
        # texts_test_path = os.path.join(self.path, "medprocner_test/txt/")

        df_train = pd.read_csv(train_path, sep="\t", dtype=str)
        # df_test = pd.read_csv(test_path, sep="\t", dtype=str)
        
        df_train["split"] = "train"
        # df_test["split"] = "test"
        
        # df = pd.concat([df_train, df_test])
        df = df_train
        
        df.rename(columns={"text": "span"}, inplace=True)
        
        df_texts = handlers.get_texts(texts_train_path)
        df = df.merge(df_texts, on="filename", how="left")
        
        self.df = df
        
        return df
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        
        d_map_names = {"label": "mention_class", "need_context": "needs_context"}
        
        self.df["filenameid"] = self.df["filename"] + "#" + self.df["start_span"] + "#" + self.df["end_span"]
        self.df.drop(columns=["filename", "start_span", "end_span"], inplace=True)

        self.df.rename(columns=d_map_names, inplace=True)
        self.df.drop(columns=["ann_id"], inplace=True)
        
        for col in self.DS_COLUMNS:
            if col not in self.df.columns:
                self.df[col] = None
        
        cols = self.DS_COLUMNS + ["text", "split"]
        self.df = self.df[cols]
        
        assert self.df.columns.intersection(self.DS_COLUMNS).shape[0] == len(self.DS_COLUMNS), "There are missing columns"
        
        return self.df
        
    def _download_data(self, download_path):
        # Ensure download path exists
        os.makedirs(download_path, exist_ok=True)

        # Download dataset
        print("Downloading dataset...")
        temp_zip_path = os.path.join(download_path, "temp_dataset.zip")
        handlers.progress_download(self.URL, temp_zip_path)

        # Extract if zip file
        with ZipFile(temp_zip_path, 'r') as zip_file:
            zip_file.extractall(download_path)
        
        # Clean up the temporary zip file
        os.remove(temp_zip_path)
        print("Dataset downloaded and extracted successfully.")

        return download_path
            
class DrugtemistGazetteer(BenchmarkDataset):
    URL = "https://zenodo.org/records/8224056/files/medprocner_gs_train+test+gazz+multilingual+crossmap_230808.zip?download=1"
    NAME = "medprocner_gs_train+test+gazz+multilingual+crossmap_230808"
    DS_COLUMNS = config.DS_COLUMNS
    
    def __init__(self, lang="es", path=None, name=NAME, url=URL, download_if_missing=True):
        super().__init__(lang, name, path, url, download_if_missing)

    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        gaz_path = os.path.join(self.path, "medprocner_gazetteer/gazzeteer_medprocner_v1_noambiguity.tsv")
        df = pd.read_csv(gaz_path, sep="\t")
        
        self.df = df
        
        return df
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        self.df = self.df[config.GZ_COLUMNS]
        
        return self.df

    def _download_data(self, download_path):
        # Ensure download path exists
        os.makedirs(download_path, exist_ok=True)

        # Download dataset
        print("Downloading dataset...")
        temp_zip_path = os.path.join(download_path, "temp_dataset.zip")
        handlers.progress_download(self.URL, temp_zip_path)

        # Extract if zip file
        with ZipFile(temp_zip_path, 'r') as zip_file:
            zip_file.extractall(download_path)
        
        # Clean up the temporary zip file
        os.remove(temp_zip_path)
        print("Dataset downloaded and extracted successfully.")

        return download_path

In [14]:
dg = Drugtemist()

preprocessing data...


In [15]:
dg.df

,filenameid,mention_class,span,code,sem_rel,is_abbreviation,is_composite,needs_context,extension_esp,text,split
0,es-S0365-66912010000700004-1#196#204,FARMACO,insulina,None,None,None,None,None,None,Mujer de 61 años con antecedentes sistémicos d...,train
1,es-S0365-66912010000700004-1#258#272,FARMACO,rosiglitiazona,None,None,None,None,None,None,Mujer de 61 años con antecedentes sistémicos d...,train
2,es-S0365-66912010000700004-1#282#289,FARMACO,Avandia,None,None,None,None,None,None,Mujer de 61 años con antecedentes sistémicos d...,train
3,es-S0365-66912010000700004-1#1116#1129,FARMACO,rosiglitazona,None,None,None,None,None,None,Mujer de 61 años con antecedentes sistémicos d...,train
4,es-S0210-48062004000500011-1#2056#2067,FARMACO,Clorambucil,None,None,None,None,None,None,En julio de 2000 y procedente del Servicio de ...,train
...,...,...,...,...,...,...,...,...,...,...,...
2773,es-S0210-56912006000200007-1#2662#2675,FARMACO,noradrenalina,None,None,None,None,None,None,Varón de 44 años con historia previa de depres...,train
2774,es-S0210-56912006000200007-1#2637#2644,FARMACO,fósforo,None,None,None,None,None,None,Varón de 44 años con historia previa de depres...,train
2775,es-S1130-01082005001200011-1#3850#3860,FARMACO,clonazepan,None,None,None,None,None,None,Varón de 58 años que acudió a su hospital de r...,train
2776,es-S1130-01082005001200011-1#3863#3872,FARMACO,piracetan,None,None,None,None,None,None,Varón de 58 años que acudió a su hospital de r...,train


In [5]:
import pandas as pd
from nlp4bia.datasets import config
df = pd.read_csv(os.path.join(config.NLP4BIA_DATA_PATH, "multicardioner_train+dev+test+bg+mappings_240528", "track2/drugtemist_train/es/tsv/multicardioner_track2_drugtemist_train_es.tsv"), sep="\t", dtype=str)

In [6]:
df

,filename,ann_id,label,start_span,end_span,text
0,es-S0365-66912010000700004-1,T13,FARMACO,196,204,insulina
1,es-S0365-66912010000700004-1,T14,FARMACO,258,272,rosiglitiazona
2,es-S0365-66912010000700004-1,T15,FARMACO,282,289,Avandia
3,es-S0365-66912010000700004-1,T16,FARMACO,1116,1129,rosiglitazona
4,es-S0210-48062004000500011-1,T17,FARMACO,2056,2067,Clorambucil
...,...,...,...,...,...,...
2773,es-S0210-56912006000200007-1,T47,FARMACO,2662,2675,noradrenalina
2774,es-S0210-56912006000200007-1,T48,FARMACO,2637,2644,fósforo
2775,es-S1130-01082005001200011-1,T78,FARMACO,3850,3860,clonazepan
2776,es-S1130-01082005001200011-1,T79,FARMACO,3863,3872,piracetan
